In [2]:
# =============================================================================
# TF-IDF + Logistic Regression — Non-Deep Baseline
# =============================================================================
# Purpose: establish the minimum performance achievable without deep learning.
# This baseline uses only bag-of-words features (TF-IDF) and a linear
# classifier — no embeddings, no sequential modeling, no context.
# Its limitations directly motivate the use of BiLSTM and Transformers.
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

SEED     = 42
VAL_SIZE = 0.1

# =============================================================================
# LOAD DATA — same cleaned source as all other notebooks
# =============================================================================
df_full = pd.read_csv('../data/augmented_train.csv')

# Keep original rows only (no augmented data for the baseline)
original_max_id = df_full['id'].max() // 2
df_orig = df_full[df_full['id'] <= original_max_id].copy()
df_orig = df_orig[['text_cleaned', 'target_relabeled']].dropna()
df_orig = df_orig.rename(columns={'target_relabeled': 'label'})
df_orig = df_orig.drop_duplicates(subset='text_cleaned')

df_test = pd.read_csv('../data/test_cleaned.csv')

print(f"Training samples : {len(df_orig)}")
print(f"Test samples     : {len(df_test)}")

# =============================================================================
# TRAIN / VALIDATION SPLIT — identical to all other notebooks
# =============================================================================
train_df, val_df = train_test_split(
    df_orig,
    test_size=VAL_SIZE,
    stratify=df_orig['label'],
    random_state=SEED
)

print(f"Train : {len(train_df)} | Val : {len(val_df)}")

# =============================================================================
# TF-IDF VECTORIZATION
# =============================================================================
# Word unigrams only — intentionally simple to establish the lower bound.
# No char n-grams, no sublinear_tf tuning. This is the true minimum baseline.
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 1),
    stop_words='english'
)

X_train = vectorizer.fit_transform(train_df['text_cleaned'])
X_val   = vectorizer.transform(val_df['text_cleaned'])
X_test  = vectorizer.transform(df_test['text_cleaned'])

y_train = train_df['label'].values
y_val   = val_df['label'].values

# =============================================================================
# LOGISTIC REGRESSION
# =============================================================================
clf = LogisticRegression(max_iter=1000, random_state=SEED)
clf.fit(X_train, y_train)

val_preds = clf.predict(X_val)
val_f1    = f1_score(y_val, val_preds)

print(f"\nValidation F1 : {val_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, val_preds,
      target_names=['Not Disaster', 'Disaster']))

# =============================================================================
# WHAT THIS TELLS US
# =============================================================================
# TF-IDF treats each tweet as a bag of words — word order is lost,
# context is ignored, and the same word gets the same representation
# regardless of surrounding text.
#
# This is precisely the limitation that motivates sequential models (BiLSTM)
# and contextual embeddings (BERTweet): disaster-related keywords like
# "fire" or "flood" appear in both real and metaphorical contexts,
# and TF-IDF cannot distinguish them.
#
# Expected Val F1: ~0.76-0.79 → clearly below BiLSTM (0.776 val) and
# Transformer models, justifying the added complexity.

# =============================================================================
# KAGGLE SUBMISSION (optional)
# =============================================================================
test_preds = clf.predict(X_test)
submission = pd.DataFrame({'id': df_test['id'], 'target': test_preds})
submission.to_csv('../data/submission_tfidf_lr.csv', index=False)
print(f"\nSubmission saved → ../data/submission_tfidf_lr.csv")
print(submission['target'].value_counts())

Training samples : 6985
Test samples     : 3263
Train : 6286 | Val : 699

Validation F1 : 0.7305

Classification Report:
              precision    recall  f1-score   support

Not Disaster       0.79      0.91      0.84       413
    Disaster       0.83      0.65      0.73       286

    accuracy                           0.80       699
   macro avg       0.81      0.78      0.79       699
weighted avg       0.81      0.80      0.80       699


Submission saved → ../data/submission_tfidf_lr.csv
target
0    2206
1    1057
Name: count, dtype: int64
